<div style="background-color:#DFC7F2; padding:15px; margin:15px;">


**Центр непрерывного образования НИУ ВШЭ**

## Программа «Python для автоматизации и анализа данных»

### Занятие 6. Часть 3. Декораторы и генераторы
*Автор: Маргарита Бурова*

# Часть 1. Декораторы

## Функции — объекты первого класса

<div style="background-color: rgba(220,255,5,1.00); padding: 15px; margin: 15px; border-radius: 8px; border-left: 5px solid rgba(220,255,5,1.00);color:black; width:80%">

В Python функции — это **объекты первого класса** (first-class objects). Это означает, что функции можно:
* присваивать переменным,
* передавать как аргументы в другие функции,
* возвращать из функций.

Именно эта особенность лежит в основе декораторов.
</div>

### Присвоение функции переменной

In [1]:
def greet(name):
    return f"Привет, {name}!"

hello = greet  # присваиваем функцию переменной (без скобок!)
print(hello("Анна"))
print(type(hello))

Привет, Анна!
<class 'function'>


### Передача функции как аргумента

In [2]:
def shout(text):
    return text.upper()

def whisper(text):
    return text.lower()

def speak(func, text):
    """Принимает функцию и применяет её к тексту"""
    return func(text)

print(speak(shout, "Привет мир"))
print(speak(whisper, "Привет мир"))

ПРИВЕТ МИР
привет мир


### Возврат функции из функции

In [3]:
def multiplier(n):
    """Возвращает функцию, которая умножает на n"""
    def inner(x):
        return x * n
    return inner

double = multiplier(2)
triple = multiplier(3)

print(double(5))   # 10
print(triple(5))   # 15

10
15


## Замыкания (closures)

<div style="background-color: rgba(220,255,5,1.00); padding: 15px; margin: 15px; border-radius: 8px; border-left: 5px solid rgba(220,255,5,1.00);color:black; width:80%">

**Замыкание** — это вложенная функция, которая «запоминает» переменные из окружающей области видимости, даже когда внешняя функция уже завершилась.

Замыкания — ключевой механизм, на котором построены декораторы.
</div>

In [4]:
def counter():
    count = 0
    def increment():
        nonlocal count  # указываем, что используем переменную из внешней функции
        count += 1
        return count
    return increment

my_counter = counter()
print(my_counter())  # 1
print(my_counter())  # 2
print(my_counter())  # 3

1
2
3


## Что такое декоратор?

<div style="background-color: rgba(220,255,5,1.00); padding: 15px; margin: 15px; border-radius: 8px; border-left: 5px solid rgba(220,255,5,1.00);color:black; width:80%">

**Декоратор** — это функция, которая принимает другую функцию и расширяет её поведение, **не изменяя** её исходный код.

Декоратор оборачивает (wrap) функцию в другую функцию.
</div>

### Декоратор «вручную»

Сначала посмотрим, как декоратор работает без специального синтаксиса:

In [5]:
def my_decorator(func):
    def wrapper():
        print("Что-то делаем ДО вызова функции")
        func()
        print("Что-то делаем ПОСЛЕ вызова функции")
    return wrapper

def say_hello():
    print("Привет!")

# Декорируем вручную
say_hello = my_decorator(say_hello)
say_hello()

Что-то делаем ДО вызова функции
Привет!
Что-то делаем ПОСЛЕ вызова функции


### Синтаксис <code style="color:red;background:none">@</code>

Python предоставляет удобный синтаксис — символ <code style="color:red;background:none">@</code> перед определением функции. Это делает ровно то же самое, что и ручное оборачивание:

In [6]:
def my_decorator(func):
    def wrapper():
        print("Что-то делаем ДО вызова функции")
        func()
        print("Что-то делаем ПОСЛЕ вызова функции")
    return wrapper

@my_decorator  # эквивалент say_hello = my_decorator(say_hello)
def say_hello():
    print("Привет!")

say_hello()

Что-то делаем ДО вызова функции
Привет!
Что-то делаем ПОСЛЕ вызова функции


## Декоратор для функций с аргументами

Чтобы декоратор работал с **любыми** функциями (с разным количеством аргументов), используем <code style="color:red;background:none">*args</code> и <code style="color:red;background:none">**kwargs</code>:

In [7]:
def logger(func):
    def wrapper(*args, **kwargs):
        print(f"Вызывается функция: {func.__name__}")
        print(f"Аргументы: {args}, {kwargs}")
        result = func(*args, **kwargs)
        print(f"Результат: {result}")
        return result
    return wrapper

@logger
def add(a, b):
    return a + b

@logger
def greet(name, greeting="Привет"):
    return f"{greeting}, {name}!"

add(3, 5)
print()
greet("Иван", greeting="Здравствуйте")

Вызывается функция: add
Аргументы: (3, 5), {}
Результат: 8

Вызывается функция: greet
Аргументы: ('Иван',), {'greeting': 'Здравствуйте'}
Результат: Здравствуйте, Иван!


'Здравствуйте, Иван!'

## Примеры декораторов на практике

### Замер времени выполнения функции

Один из самых популярных декораторов — замер времени работы функции:

In [8]:
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"Функция {func.__name__} выполнялась {end - start:.4f} сек.")
        return result
    return wrapper

@timer
def slow_function():
    time.sleep(1)
    return "Готово!"

slow_function()

Функция slow_function выполнялась 1.0005 сек.


'Готово!'

### Декоратор для подсчёта вызовов

In [9]:
def count_calls(func):
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        print(f"Вызов #{wrapper.calls} функции {func.__name__}")
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

@count_calls
def say_hi():
    print("Привет!")

say_hi()
say_hi()
say_hi()
print(f"Всего вызовов: {say_hi.calls}")

Вызов #1 функции say_hi
Привет!
Вызов #2 функции say_hi
Привет!
Вызов #3 функции say_hi
Привет!
Всего вызовов: 3


## <code style="color:red;background:none">functools.wraps</code>

<div style="background-color: rgba(220,255,5,1.00); padding: 15px; margin: 15px; border-radius: 8px; border-left: 5px solid rgba(220,255,5,1.00);color:black; width:80%">

При декорировании функция «теряет» своё имя и документацию — вместо них отображается имя обёртки (`wrapper`). Чтобы это исправить, используют <code style="color:red;background:none">functools.wraps</code>.
</div>

In [10]:
from functools import wraps

def smart_decorator(func):
    @wraps(func)  # сохраняем метаданные оригинальной функции
    def wrapper(*args, **kwargs):
        """Это обёртка"""
        return func(*args, **kwargs)
    return wrapper

@smart_decorator
def my_function():
    """Это моя функция"""
    pass

print(my_function.__name__)  # my_function, а не wrapper
print(my_function.__doc__)   # Это моя функция

my_function
Это моя функция


## Декоратор с параметрами

Иногда нужно передать параметры **самому декоратору**. Для этого создаётся ещё один уровень вложенности — «фабрика декораторов»:

In [11]:
from functools import wraps

def repeat(n):
    """Фабрика декораторов: повторяет вызов функции n раз"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)  # повторить 3 раза
def say(message):
    print(message)

say("Ура!")

Ура!
Ура!
Ура!


## Цепочка декораторов

На одну функцию можно повесить несколько декораторов. Они применяются **снизу вверх**:

In [12]:
def bold(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return f"<b>{func(*args, **kwargs)}</b>"
    return wrapper

def italic(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return f"<i>{func(*args, **kwargs)}</i>"
    return wrapper

@bold       # 2) оборачиваем в <b>
@italic     # 1) сначала оборачиваем в <i>
def greet(name):
    return f"Привет, {name}"

print(greet("Мир"))  # <b><i>Привет, Мир</i></b>

<b><i>Привет, Мир</i></b>


# Часть 2. Генераторы

## Вспомним: итераторы и итерируемые объекты

<div style="background-color: rgba(220,255,5,1.00); padding: 15px; margin: 15px; border-radius: 8px; border-left: 5px solid rgba(220,255,5,1.00);color:black; width:80%">

**Итерируемый объект (iterable)** — объект, по которому можно пройти в цикле `for` (списки, строки, словари, файлы и т.д.).

**Итератор (iterator)** — объект, который выдаёт элементы по одному с помощью функции <code style="color:red;background:none">next()</code>.

</div>

In [13]:
my_list = [10, 20, 30]
my_iter = iter(my_list)  # создаём итератор из списка

print(next(my_iter))  # 10
print(next(my_iter))  # 20
print(next(my_iter))  # 30
# print(next(my_iter))  # StopIteration — элементы закончились!

10
20
30


## Что такое генератор?

<div style="background-color: rgba(220,255,5,1.00); padding: 15px; margin: 15px; border-radius: 8px; border-left: 5px solid rgba(220,255,5,1.00);color:black; width:80%">

**Генератор** — это специальный вид итератора, который создаётся с помощью функции с ключевым словом <code style="color:red;background:none">yield</code>.

Главное отличие от обычной функции:
* обычная функция вычисляет **все** значения сразу и возвращает их,
* генератор вычисляет значения **по одному** (лениво) — каждый раз при вызове `next()` он «просыпается», выполняет код до следующего `yield` и «засыпает».

Это экономит память, особенно при работе с большими объёмами данных.
</div>

### Простейший генератор

In [15]:
def my_generator():
    print("Первое значение")
    yield 1
    print("Второе значение")
    yield 2
    print("Третье значение")
    yield 3

gen = my_generator()
print(type(gen))  # <class 'generator'>

<class 'generator'>


In [16]:
print(next(gen))  # Первое значение -> 1
print(next(gen))  # Второе значение -> 2
print(next(gen))  # Третье значение -> 3

Первое значение
1
Второе значение
2
Третье значение
3


Генератор можно использовать в цикле <code style="color:red;background:none">for</code> — он автоматически обрабатывает `StopIteration`:

In [17]:
for value in my_generator():
    print(value)

Первое значение
1
Второе значение
2
Третье значение
3


## <code style="color:red;background:none">yield</code> vs <code style="color:red;background:none">return</code>

| Свойство | `return` | `yield` |
|----------|----------|---------|
| Завершает функцию? | Да | Нет, приостанавливает |
| Сколько значений? | Одно | Много (по одному за вызов) |
| Состояние | Теряется | Сохраняется |
| Тип результата | Обычный объект | Генератор |

In [18]:
# С return — всё сразу в памяти
def get_squares_list(n):
    result = []
    for i in range(n):
        result.append(i ** 2)
    return result

# С yield — по одному элементу
def get_squares_gen(n):
    for i in range(n):
        yield i ** 2

print(get_squares_list(5))           # [0, 1, 4, 9, 16]
print(list(get_squares_gen(5)))      # [0, 1, 4, 9, 16]

[0, 1, 4, 9, 16]
[0, 1, 4, 9, 16]


## Экономия памяти

Генераторы особенно полезны при работе с большими данными. Сравним использование памяти:

In [19]:
import sys

# Список из миллиона элементов
big_list = [i ** 2 for i in range(1_000_000)]
print(f"Список: {sys.getsizeof(big_list):,} байт")

# Генератор из миллиона элементов
big_gen = (i ** 2 for i in range(1_000_000))
print(f"Генератор: {sys.getsizeof(big_gen):,} байт")

Список: 8,448,728 байт
Генератор: 200 байт


## Генераторные выражения (generator expressions)

<div style="background-color: rgba(220,255,5,1.00); padding: 15px; margin: 15px; border-radius: 8px; border-left: 5px solid rgba(220,255,5,1.00);color:black; width:80%">

Точно так же, как **списковые включения** (list comprehensions) создают списки, **генераторные выражения** создают генераторы. Разница — в скобках:
* `[x**2 for x in range(10)]` — список (квадратные скобки)
* `(x**2 for x in range(10))` — генератор (круглые скобки)
</div>

In [20]:
# Списковое включение
squares_list = [x**2 for x in range(10)]
print(type(squares_list), squares_list)

# Генераторное выражение
squares_gen = (x**2 for x in range(10))
print(type(squares_gen))
print(list(squares_gen))  # конвертируем, чтобы увидеть значения

<class 'list'> [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
<class 'generator'>
[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]


Генераторные выражения удобно передавать прямо в функции:

In [21]:
# Сумма квадратов первых 100 чисел — без создания промежуточного списка
total = sum(x**2 for x in range(100))
print(total)

328350


## Бесконечные генераторы

Генераторы могут быть **бесконечными** — они будут выдавать значения до тех пор, пока мы их запрашиваем:

In [22]:
def infinite_counter(start=0):
    """Бесконечный счётчик"""
    n = start
    while True:
        yield n
        n += 1

counter = infinite_counter(1)
print(next(counter))  # 1
print(next(counter))  # 2
print(next(counter))  # 3
print(next(counter))  # 4

1
2
3
4


In [23]:
def fibonacci():
    """Бесконечный генератор чисел Фибоначчи"""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

fib = fibonacci()
# Возьмём первые 10 чисел Фибоначчи
first_10 = [next(fib) for _ in range(10)]
print(first_10)

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


## <code style="color:red;background:none">yield from</code>

Конструкция <code style="color:red;background:none">yield from</code> позволяет делегировать генерацию значений другому итерируемому объекту или генератору:

In [24]:
def gen_1():
    yield 1
    yield 2

def gen_2():
    yield 3
    yield 4

def combined():
    yield from gen_1()  # делегируем генерацию
    yield from gen_2()

print(list(combined()))  # [1, 2, 3, 4]

[1, 2, 3, 4]


In [25]:
# yield from работает с любым итерируемым объектом
def flatten(nested_list):
    """Разворачивает вложенный список"""
    for item in nested_list:
        if isinstance(item, list):
            yield from flatten(item)  # рекурсия!
        else:
            yield item

nested = [1, [2, 3], [4, [5, 6]], 7]
print(list(flatten(nested)))  # [1, 2, 3, 4, 5, 6, 7]

[1, 2, 3, 4, 5, 6, 7]


## Метод <code style="color:red;background:none">send()</code>

Генераторы могут не только **отдавать** значения, но и **принимать** их через метод <code style="color:red;background:none">send()</code>:

In [26]:
def accumulator():
    """Генератор-накопитель: принимает числа и возвращает текущую сумму"""
    total = 0
    while True:
        value = yield total
        if value is not None:
            total += value

acc = accumulator()
next(acc)         # инициализация — обязательно!
print(acc.send(10))  # 10
print(acc.send(20))  # 30
print(acc.send(5))   # 35

10
30
35


## Полезные генераторы из модуля <code style="color:red;background:none">itertools</code>

Модуль <code style="color:red;background:none">itertools</code> содержит множество готовых генераторов для комбинаторики, группировки и обработки данных:

In [27]:
import itertools

# count — бесконечный счётчик
for i in itertools.islice(itertools.count(10, 2), 5):
    print(i, end=" ")  # 10 12 14 16 18
print()

# cycle — бесконечное повторение
colors = itertools.cycle(["красный", "зелёный", "синий"])
for _ in range(6):
    print(next(colors), end=" ")
print()

# chain — объединение итераторов
combined = itertools.chain([1, 2], [3, 4], [5])
print(list(combined))

10 12 14 16 18 
красный зелёный синий красный зелёный синий 
[1, 2, 3, 4, 5]


## Бонус: декоратор для генератора

Декораторы и генераторы прекрасно работают вместе. Например, можно написать декоратор, ограничивающий количество элементов генератора:

In [28]:
from functools import wraps

def limit(max_items):
    """Декоратор: ограничивает генератор до max_items элементов"""
    def decorator(gen_func):
        @wraps(gen_func)
        def wrapper(*args, **kwargs):
            gen = gen_func(*args, **kwargs)
            for _, value in zip(range(max_items), gen):
                yield value
        return wrapper
    return decorator

@limit(5)
def natural_numbers():
    """Бесконечный генератор натуральных чисел"""
    n = 1
    while True:
        yield n
        n += 1

print(list(natural_numbers()))  # [1, 2, 3, 4, 5]

[1, 2, 3, 4, 5]


## Потренируемся!

**№1.** Напишите декоратор <code style="color:red;background:none">uppercase_result</code>, который преобразует строковый результат любой функции в верхний регистр.

Пример:
```python
@uppercase_result
def greet(name):
    return f"привет, {name}"

print(greet("Мир"))  # ПРИВЕТ, МИР
```

**№2.** Напишите декоратор <code style="color:red;background:none">cache</code>, который запоминает результаты вызовов функции и при повторном вызове с теми же аргументами возвращает кэшированное значение (без повторного вычисления). Выведите сообщение, если значение берётся из кэша.

Пример:
```python
@cache
def slow_square(x):
    time.sleep(1)
    return x ** 2

print(slow_square(4))  # ждёт 1 сек, возвращает 16
print(slow_square(4))  # мгновенно, из кэша — 16
```

**№3.** Напишите декоратор <code style="color:red;background:none">validate_positive</code>, который проверяет, что **все** числовые аргументы функции положительные. Если нет — вызывает <code style="color:red;background:none">ValueError</code>.

Пример:
```python
@validate_positive
def multiply(a, b):
    return a * b

print(multiply(3, 5))   # 15
print(multiply(-1, 5))  # ValueError!
```

**№4.** Напишите генератор <code style="color:red;background:none">even_numbers(n)</code>, который выдаёт первые `n` чётных чисел (начиная с 2).

Пример:
```python
print(list(even_numbers(5)))  # [2, 4, 6, 8, 10]
```

**№5.** Напишите генератор <code style="color:red;background:none">running_average()</code>, который принимает числа через <code style="color:red;background:none">send()</code> и выдаёт текущее среднее арифметическое.

Пример:
```python
avg = running_average()
next(avg)             # инициализация
print(avg.send(10))   # 10.0
print(avg.send(20))   # 15.0
print(avg.send(30))   # 20.0
```

**№6.** Напишите генераторное выражение, которое из списка слов оставляет только те, длина которых больше 4 символов, и переводит их в верхний регистр.

Пример:
```python
words = ["кот", "генератор", "дом", "python", "яблоко", "ум"]
# Результат: ['ГЕНЕРАТОР', 'PYTHON', 'ЯБЛОКО']
```

**№7.** Напишите генератор <code style="color:red;background:none">file_reader(filepath)</code>, который читает файл построчно и выдаёт только непустые строки (без символов перевода строки). Это типичный паттерн для обработки больших файлов без загрузки в память.

Пример:
```python
for line in file_reader("data.txt"):
    print(line)
```

**№8.** Напишите декоратор <code style="color:red;background:none">retry(max_attempts)</code>, который при возникновении исключения повторяет вызов функции до `max_attempts` раз. Если все попытки неудачны — пробрасывает последнее исключение.

Пример:
```python
import random

@retry(max_attempts=5)
def unreliable():
    if random.random() < 0.7:
        raise ConnectionError("Сбой соединения")
    return "Успех!"

print(unreliable())
```